In [1]:
import pandas as pd
import torch 
import numpy as np
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
# Load the extracted data points
df = pd.read_csv('gestures.csv')

X = df.drop('Labels', axis=1).values
y = df['Labels'].values

num_classes = len(np.unique(y))
print(f'Total data row: {len(df)} with {num_classes} gesture classes')
print(f'Feature dimensions: {X.shape[1]}')
print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

Total data row: 3064 with 6 gesture classes
Feature dimensions: 42
Class distribution: {0: 500, 1: 510, 2: 492, 3: 498, 4: 529, 5: 535}


In [3]:
# Train, Test and Split (Train: 80%, Test: 20%)
train_X, test_X, train_y, test_y = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize the data points
scaler = StandardScaler()

train_X = scaler.fit_transform(train_X)
test_X = scaler.transform(test_X)

# Clip extreme outliers (past 5 std devs)
train_X = np.clip(train_X, -5.0, 5.0)
test_X = np.clip(test_X, -5.0, 5.0)

# Convert to pytorch tensor
train_X_t = torch.FloatTensor(train_X)
test_X_t = torch.FloatTensor(test_X)
train_y_t = torch.LongTensor(train_y)
test_y_t = torch.LongTensor(test_y)

In [4]:
# Define Lightweight 3-layer ANN 
class GestureANN(nn.Module):
    def __init__(self, input_dim=42, num_classes=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.net(x)


model = GestureANN(input_dim=42, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [5]:
# Train Model
epochs = 60
print('-----Training Model------')

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(train_X_t)
    loss = criterion(outputs, train_y_t)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}] - Loss: {loss.item():.4f}')

-----Training Model------
Epoch [10/60] - Loss: 1.5787
Epoch [20/60] - Loss: 1.3499
Epoch [30/60] - Loss: 1.0926
Epoch [40/60] - Loss: 0.8385
Epoch [50/60] - Loss: 0.6093
Epoch [60/60] - Loss: 0.4237


In [6]:
# Evaluate Accuracy
model.eval()
with torch.no_grad():
    test_outputs = model(test_X_t)
    _, predicted = torch.max(test_outputs, 1)
    correct = (predicted == test_y_t).sum().item()
    accuracy = (correct / len(test_y_t)) * 100
    print(f"\n✅ Final Test Accuracy: {accuracy:.2f}%")

    # Confusion matrix
    cm = confusion_matrix(test_y_t.numpy(), predicted.numpy())
    print(f"\nConfusion Matrix:")
    class_names = [f"Class {i}" for i in range(num_classes)]
    print(f"          {'  '.join(class_names)}")
    for i, row in enumerate(cm):
        print(f"  {class_names[i]:>8} {row}")
    print(f"\nClassification Report:")
    print(classification_report(test_y_t.numpy(), predicted.numpy(),
                                target_names=class_names, digits=4))

    # Per-class accuracy
    print("\nPer-class accuracy:")
    for i in range(num_classes):
        mask = test_y_t == i
        if mask.sum() > 0:
            class_acc = (predicted[mask] == test_y_t[mask]).float().mean().item() * 100
            print(f"  Class {i}: {class_acc:.1f}% ({mask.sum()} samples)")

# ── Code cell 7 ──
# Save Model Checkpoint
torch.save({
    'model_state': model.state_dict(),
    'scaler': scaler,
    'num_classes': num_classes,
}, "gesture_ann_model.pth")
print("💾 Model saved successfully as 'gesture_ann_model.pth'!")


✅ Final Test Accuracy: 95.27%

Confusion Matrix:
          Class 0  Class 1  Class 2  Class 3  Class 4  Class 5
   Class 0 [100   0   0   0   0   0]
   Class 1 [  0 102   0   0   0   0]
   Class 2 [ 0  0 98  0  0  0]
   Class 3 [  0   0   0 100   0   0]
   Class 4 [  0   0   0   0 106   0]
   Class 5 [27  0  0  2  0 78]

Classification Report:
              precision    recall  f1-score   support

     Class 0     0.7874    1.0000    0.8811       100
     Class 1     1.0000    1.0000    1.0000       102
     Class 2     1.0000    1.0000    1.0000        98
     Class 3     0.9804    1.0000    0.9901       100
     Class 4     1.0000    1.0000    1.0000       106
     Class 5     1.0000    0.7290    0.8432       107

    accuracy                         0.9527       613
   macro avg     0.9613    0.9548    0.9524       613
weighted avg     0.9621    0.9527    0.9516       613


Per-class accuracy:
  Class 0: 100.0% (100 samples)
  Class 1: 100.0% (102 samples)
  Class 2: 100.0% (98 sam